In [2]:
# BASE 3 — df_pacientes_trayectorias (paciente)
    # Objetivo

        # Tener una historia coherente por paciente

# ACÁ SÍ entra el problema difícil: consistencia


# Un paciente es consistente si:

# Nivel básico (te recomiendo este)
    # tiene ≥1 episodio válido
    # fechas ordenables
    # edad consistente (tu regla está bien)
    # tiene un final razonable
# Nivel intermedio (ideal para vos)
    # el último evento válido define el desenlace
    # no hay eventos después de una defunción
    # la trayectoria temporal no tiene saltos absurdos

In [4]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import ast
import pandas as pd
import numpy as np
from src.config import *

In [ ]:
df_base = pd.read_excel("../data/final_data/df_base_limpia.xlsx")
df_traslados = pd.read_excel("../data/final_data/df_traslados_strict.xlsx")

In [8]:
# 0. PREPARACION
# ==============================================================================

# ordenar base
df_base = df_base.sort_values(['paciente_id', 'fecha_ingreso']).copy()


# ==============================================================================

In [9]:
# 1. FILTROS ESTRICTOS A NIVEL PACIENTE
# ==============================================================================

# --- fechas completas
df_base['flag_fechas_incompletas'] = (
    df_base['fecha_ingreso'].isna() |
    df_base['fecha_egreso'].isna()
)

# --- tolerancia temporal (minutos)
TOL_MINUTOS = 10

df_base['delta_min'] = (
    (df_base['fecha_egreso'] - df_base['fecha_ingreso'])
    .dt.total_seconds() / 60
)

df_base['flag_fechas_invalidas'] = (
    df_base['delta_min'] < -TOL_MINUTOS
)

# --- consistencia edad
df_base['edad_next'] = df_base.groupby('paciente_id')['edad'].shift(-1)

df_base['delta_edad'] = df_base['edad_next'] - df_base['edad']

df_base['flag_edad_inconsistente'] = (
    df_base['delta_edad'].abs() > 2
)

# --- egreso administrativo final
idx_last = df_base.groupby('paciente_id')['fecha_ingreso'].idxmax()

df_last = df_base.loc[idx_last, ['paciente_id', 'tipo_egreso']].copy()

df_last['flag_fin_admin'] = df_last['tipo_egreso'] == 'administrativo'


# --- agregación paciente
df_flags_paciente = df_base.groupby('paciente_id').agg({
    'flag_fechas_incompletas': 'max',
    'flag_fechas_invalidas': 'max',
    'flag_edad_inconsistente': 'max'
}).reset_index()

df_flags_paciente = df_flags_paciente.merge(
    df_last[['paciente_id', 'flag_fin_admin']],
    on='paciente_id',
    how='left'
)

# --- paciente válido
df_flags_paciente['paciente_valido'] = (
    ~df_flags_paciente['flag_fechas_incompletas'] &
    ~df_flags_paciente['flag_fechas_invalidas'] &
    ~df_flags_paciente['flag_edad_inconsistente'] &
    ~df_flags_paciente['flag_fin_admin']
)

pacientes_validos = df_flags_paciente.loc[
    df_flags_paciente['paciente_valido'],
    'paciente_id'
]


# --- base filtrada
df_base_filtrada = df_base[
    df_base['paciente_id'].isin(pacientes_validos)
].copy()


# ==============================================================================

In [10]:
# 2. ANCLAR EN TRASLADOS
# ==============================================================================

df_traslados_filtrado = df_traslados[
    df_traslados['paciente_id'].isin(pacientes_validos)
].copy()

df_traslados_filtrado = df_traslados_filtrado.sort_values(
    ['paciente_id', 'fecha_egreso_origen']
)


# separar pacientes
pacientes_con_traslado = df_traslados_filtrado['paciente_id'].unique()
pacientes_sin_traslado = list(set(pacientes_validos) - set(pacientes_con_traslado))


# ==============================================================================


In [11]:
# 3. CONSTRUCCION DE TRAYECTORIAS (PACIENTES CON TRASLADOS)
# ==============================================================================

def construir_trayectoria(grp):
    grp = grp.sort_values('fecha_egreso_origen')
    
    hospitales = [grp.iloc[0]['hospital_origen']]
    
    for _, row in grp.iterrows():
        hospitales.append(row['hospital_destino'])
    
    return pd.Series({
        'trayectoria_hospitalaria': hospitales,
        'hospital_inicio': hospitales[0],
        'hospital_final': hospitales[-1],
        'n_hospitales_unicos': len(set(hospitales))
    })


df_trayectorias_conectadas = (
    df_traslados_filtrado
    .groupby('paciente_id')
    .apply(construir_trayectoria)
    .reset_index()
)


# ==============================================================================


In [12]:
# 4. TRAYECTORIAS TRIVIALES (SIN TRASLADOS)
# ==============================================================================

df_triviales = df_base_filtrada[
    df_base_filtrada['paciente_id'].isin(pacientes_sin_traslado)
].copy()

def trayectoria_trivial(grp):
    hospitales = grp.sort_values('fecha_ingreso')['hospital_origen'].tolist()
    
    return pd.Series({
        'trayectoria_hospitalaria': hospitales,
        'hospital_inicio': hospitales[0],
        'hospital_final': hospitales[-1],
        'n_hospitales_unicos': len(set(hospitales))
    })


df_trayectorias_triviales = (
    df_triviales
    .groupby('paciente_id')
    .apply(trayectoria_trivial)
    .reset_index()
)


# ==============================================================================

In [13]:
# 5. UNIFICACION
# ==============================================================================

df_trayectorias = pd.concat([
    df_trayectorias_conectadas,
    df_trayectorias_triviales
], ignore_index=True)


# ==============================================================================


In [14]:
# 6. METRICAS BASICAS
# ==============================================================================

df_metrics = df_base_filtrada.groupby('paciente_id').agg({
    'fecha_ingreso': 'min',
    'fecha_egreso': 'max',
    'hospital_origen': 'count'
}).rename(columns={
    'fecha_ingreso': 'fecha_primer_ingreso',
    'fecha_egreso': 'fecha_ultimo_egreso',
    'hospital_origen': 'n_episodios'
}).reset_index()

df_metrics['duracion_total'] = (
    df_metrics['fecha_ultimo_egreso'] - df_metrics['fecha_primer_ingreso']
).dt.days


# ==============================================================================
# 7. DESENLACE (ROBUSTO)
# ==============================================================================

def calcular_desenlace(grp):
    grp = grp.sort_values('fecha_ingreso')
    
    muerte = grp[grp['tipo_egreso'] == 'muerte']
    
    if len(muerte) > 0:
        return pd.Series({
            'desenlace': 'muerte',
            'fecha_desenlace': muerte.iloc[0]['fecha_egreso']
        })
    
    # fallback: usar secuencia completa (no solo último)
    ultimo = grp.iloc[-1]
    
    return pd.Series({
        'desenlace': ultimo['tipo_egreso'],
        'fecha_desenlace': ultimo['fecha_egreso']
    })


df_desenlace = (
    df_base_filtrada
    .groupby('paciente_id')
    .apply(calcular_desenlace)
    .reset_index()
)


# ==============================================================================
# 8. MERGE FINAL
# ==============================================================================

df_pacientes_trayectorias = (
    df_trayectorias
    .merge(df_metrics, on='paciente_id', how='left')
    .merge(df_desenlace, on='paciente_id', how='left')
)


# ==============================================================================
# 9. CHECKS
# ==============================================================================

print("Pacientes finales:", df_pacientes_trayectorias['paciente_id'].nunique())
print("Promedio episodios:", df_pacientes_trayectorias['n_episodios'].mean())

print("\n% con múltiples hospitales:")
print((df_pacientes_trayectorias['n_hospitales_unicos'] > 1).mean())

Pacientes finales: 19980
Promedio episodios: 1.1047547547547547

% con múltiples hospitales:
0.0916916916916917


### Preguntas sobre criterios de trayectorias. Muy importante:

In [15]:
# ==============================================================================
# FUNCION ROBUSTA DE TRAYECTORIA (DESDE TRASLADOS)
# ==============================================================================

def construir_trayectoria_robusta(grp):
    
    # ordenar por tiempo del traslado (criterio: orden temporal principal)
    grp = grp.sort_values('fecha_egreso_origen').copy()
    
    
    # ------------------------------------------------------------------------------
    # INICIALIZACION
    # ------------------------------------------------------------------------------
    
    hospitales = []  # secuencia final de hospitales
    flags = {
        'flag_salto_inconsistente': 0,     # criterio: origen no coincide con destino previo
        'flag_loop': 0,                   # criterio: vuelve a hospital anterior
        'flag_hospital_repetido': 0,      # criterio: A → A
        'flag_trayectoria_cortada': 0     # criterio: se reinicia path
    }
    
    
    # ------------------------------------------------------------------------------
    # PRIMER NODO
    # ------------------------------------------------------------------------------
    
    # criterio: la trayectoria empieza en el primer hospital_origen observado
    current_hospital = grp.iloc[0]['hospital_origen']
    
    hospitales.append(current_hospital)
    
    
    # ------------------------------------------------------------------------------
    # RECORRIDO DE TRASLADOS
    # ------------------------------------------------------------------------------
    
    for _, row in grp.iterrows():
        
        origen = row['hospital_origen']
        destino = row['hospital_destino']
        
        
        # --------------------------------------------------------------------------
        # CRITERIO 1: CONTINUIDAD DEL PATH
        # --------------------------------------------------------------------------
        
        # si el origen no coincide con el último hospital de la trayectoria
        if origen != current_hospital:
            
            flags['flag_salto_inconsistente'] = 1
            
            # OPCION A (actual): cortar trayectoria y reiniciar
            # hospitales.append(origen)
            # current_hospital = origen
            
            # OPCION B: ignorar este traslado (NO hacemos nada)
            
            # OPCION C: forzar continuidad (lo dejamos como está)
            
            # 👉 por ahora: NO modificamos la trayectoria, solo flag
        
        
        # --------------------------------------------------------------------------
        # CRITERIO 2: HOSPITAL REPETIDO (A → A)
        # --------------------------------------------------------------------------
        
        if destino == current_hospital:
            flags['flag_hospital_repetido'] = 1
            
            # OPCION: ignorar este paso (no lo agregamos)
            continue
        
        
        # --------------------------------------------------------------------------
        # CRITERIO 3: LOOP (volver a un hospital previo)
        # --------------------------------------------------------------------------
        
        if destino in hospitales:
            flags['flag_loop'] = 1
        
        
        # --------------------------------------------------------------------------
        # CRITERIO 4: AGREGAR DESTINO
        # --------------------------------------------------------------------------
        
        hospitales.append(destino)
        current_hospital = destino
    
    
    # ------------------------------------------------------------------------------
    # CRITERIO 5: LONGITUD MINIMA
    # ------------------------------------------------------------------------------
    
    # si después de todo quedó 1 solo nodo → trayectoria trivial disfrazada
    if len(hospitales) == 1:
        flags['flag_trayectoria_cortada'] = 1
    
    
    # ------------------------------------------------------------------------------
    # CRITERIO 6: COLAPSAR DUPLICADOS CONSECUTIVOS
    # ------------------------------------------------------------------------------
    
    hospitales_limpios = [hospitales[0]]
    
    for h in hospitales[1:]:
        if h != hospitales_limpios[-1]:  # criterio: eliminar A → A consecutivo
            hospitales_limpios.append(h)
    
    
    # ------------------------------------------------------------------------------
    # OUTPUT
    # ------------------------------------------------------------------------------
    
    return pd.Series({
        'trayectoria_hospitalaria': hospitales_limpios,   # secuencia final
        'hospital_inicio': hospitales_limpios[0],         # primer nodo
        'hospital_final': hospitales_limpios[-1],         # último nodo
        'n_hospitales_unicos': len(set(hospitales_limpios)),  # diversidad
        
        # flags
        **flags
    })

In [ ]:
df_trayectorias_conectadas = (
    df_traslados_filtrado
    .groupby('paciente_id')
    .apply(construir_trayectoria_robusta)
    .reset_index()
)

In [ ]:
# ¿Qué hacer con saltos inconsistentes?
'''
A → B
C → D
'''


# ¿Permitir loops?
'''A → B → A'''


# ¿Eliminar A → A?
'''A → A → B'''




'A → B → A'